In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import sys
import platform
import string
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# ==========================================================
# UNIVERSAL FILE / FOLDER FINDER (with tree-style output)
# Works unmodified on: local Windows/Linux/Mac, Kaggle notebooks,
# and Google Colab. Searches recursively for a filename or partial
# filename match, then prints a full tree diagram for each match.
#
# NOTE ON GPU: File search is I/O-bound (disk/filesystem reads),
# not compute-bound, so a GPU provides no speedup here -- there's
# no matrix/tensor math to accelerate. Instead, this script uses
# multithreading to search multiple drives/roots in parallel,
# which is the actual bottleneck-relevant optimization.
# ==========================================================


def detect_environment():
    """Detect which environment we're running in."""
    if os.path.exists("/kaggle"):
        return "kaggle"
    if os.path.exists("/content"):
        return "colab"
    return "local"


def get_search_roots(env):
    """Return existing (path, label) search roots for the environment."""
    roots = []

    if env == "kaggle":
        candidates = [
            ("/kaggle/input", "Kaggle attached datasets"),
            ("/kaggle/working", "Kaggle working directory"),
        ]
        roots = [(p, label) for p, label in candidates if os.path.exists(p)]

    elif env == "colab":
        candidates = [
            ("/content", "Colab local storage"),
            ("/content/drive", "Google Drive (if mounted)"),
        ]
        roots = [(p, label) for p, label in candidates if os.path.exists(p)]

    else:  # local
        system = platform.system()
        if system == "Windows":
            for letter in string.ascii_uppercase:
                drive = f"{letter}:\\"
                if os.path.exists(drive):
                    roots.append((drive, f"Drive {letter}:"))
        else:
            candidates = [
                (str(Path.home()), "Home directory"),
                ("/mnt", "Mounted volumes (/mnt)"),
                ("/media", "Removable media (/media)"),
                ("/Volumes", "Mounted volumes (/Volumes, macOS)"),
                ("/", "Entire filesystem (/)"),
            ]
            roots = [(p, label) for p, label in candidates if os.path.exists(p)]

    return roots


def search_path(root, query, case_sensitive=False):
    """
    Recursively walk `root` for files/folders whose name contains
    `query`. Skips unreadable directories instead of crashing.
    """
    matches = []
    query_cmp = query if case_sensitive else query.lower()

    for dirpath, dirnames, filenames in os.walk(root, onerror=lambda e: None):
        for d in dirnames:
            name_cmp = d if case_sensitive else d.lower()
            if query_cmp in name_cmp:
                matches.append(os.path.join(dirpath, d))

        for f in filenames:
            name_cmp = f if case_sensitive else f.lower()
            if query_cmp in name_cmp:
                matches.append(os.path.join(dirpath, f))

        try:
            dirnames[:] = [
                d for d in dirnames
                if os.access(os.path.join(dirpath, d), os.R_OK)
            ]
        except OSError:
            dirnames[:] = []

    return matches


def search_roots_parallel(roots, query, case_sensitive=False, max_workers=None):
    """
    Search multiple roots (drives/folders) CONCURRENTLY using a thread
    pool. This is the real-world equivalent of 'using your hardware' --
    it lets the OS overlap disk reads across drives instead of doing
    them one at a time. max_workers defaults to one thread per root
    (capped reasonably) since threads are cheap for I/O-bound work.
    """
    if max_workers is None:
        max_workers = min(len(roots), 8) or 1

    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_root = {
            executor.submit(search_path, root, query, case_sensitive): (root, label)
            for root, label in roots
        }
        for future in as_completed(future_to_root):
            root, label = future_to_root[future]
            try:
                results[root] = future.result()
            except Exception as e:
                print(f"  !! Could not search {root}: {type(e).__name__}: {e}")
                results[root] = []

    return results


def _print_subtree(directory, prefix, max_entries_per_dir=None):
    """Recursively print a directory's full contents as a tree."""
    try:
        entries = sorted(os.listdir(directory))
    except (PermissionError, OSError):
        print(prefix + "└── [permission denied]")
        return

    total = len(entries)
    truncated = False
    if max_entries_per_dir is not None and total > max_entries_per_dir:
        entries = entries[:max_entries_per_dir]
        truncated = True

    for idx, entry in enumerate(entries):
        full_path = os.path.join(directory, entry)
        is_last = (idx == len(entries) - 1) and not truncated
        connector = "└── " if is_last else "├── "
        print(prefix + connector + entry)

        if os.path.isdir(full_path):
            extension = "    " if is_last else "│   "
            _print_subtree(full_path, prefix + extension, max_entries_per_dir)

    if truncated:
        remaining = total - max_entries_per_dir
        print(prefix + f"└── ... ({remaining} more item(s) not shown)")


def print_full_tree(path, max_entries_per_dir=None):
    """
    Print the ancestor chain (root -> match) then, if the match is a
    directory, recursively expand its ENTIRE subtree beneath it.
    """
    parts = Path(path).parts

    print(parts[0])
    for i in range(1, len(parts)):
        indent = "    " * (i - 1)
        print(f"{indent}└── {parts[i]}")

    if os.path.isdir(path):
        base_indent = "    " * len(parts)
        _print_subtree(path, base_indent, max_entries_per_dir)

    print()


def run_search(query, roots, case_sensitive=False, max_entries_per_dir=None, parallel=True):
    """
    Search all given roots and print tree-style results.
    parallel=True searches all roots concurrently (recommended --
    this is the actual speed optimization available on your machine).
    """
    print("\n" + "=" * 72)
    print(f"SEARCHING FOR: '{query}'  (case-{'sensitive' if case_sensitive else 'insensitive'})")
    print(f"Mode: {'parallel (multi-threaded)' if parallel else 'sequential'}")
    print("=" * 72)

    if parallel and len(roots) > 1:
        all_matches = search_roots_parallel(roots, query, case_sensitive)
    else:
        all_matches = {}
        for root, label in roots:
            try:
                all_matches[root] = search_path(root, query, case_sensitive)
            except Exception as e:
                print(f"  !! Could not search {root}: {type(e).__name__}: {e}")
                all_matches[root] = []

    for root, label in roots:
        matches = all_matches.get(root, [])
        print(f"\n--- {label} ({root}) ---")
        if matches:
            print(f"  Found {len(matches)} match(es):\n")
            for m in matches:
                print_full_tree(m, max_entries_per_dir)
        else:
            print(f"  No matches found under {root}.")

    total = sum(len(v) for v in all_matches.values())
    print("=" * 72)
    print(f"SEARCH COMPLETE: {total} total match(es) across {len(roots)} location(s).")
    print("=" * 72)

    return all_matches


def interactive_search():
    """Interactive terminal loop for local use."""
    env = detect_environment()
    print(f"Detected environment: {env.upper()}")

    all_roots = get_search_roots(env)
    if not all_roots:
        print("No accessible search roots were found in this environment.")
        return

    print("\nAvailable search locations:")
    for i, (path, label) in enumerate(all_roots, 1):
        print(f"  [{i}] {label}  ->  {path}")

    while True:
        query = input("\nEnter filename or partial filename to search for (or 'quit'): ").strip()
        if query.lower() in ("quit", "exit", "q"):
            print("Exiting.")
            break
        if not query:
            print("Please enter a non-empty search term.")
            continue

        case_sensitive = input("Case-sensitive search? (y/N): ").strip().lower() == "y"

        print("\nSearch scope:")
        print("  [a] All available locations")
        print("  [n] Pick specific location(s) by number")
        scope = input("Choose scope (a/n): ").strip().lower()

        if scope == "n":
            selection = input(f"Enter location numbers separated by commas (1-{len(all_roots)}): ").strip()
            try:
                indices = [int(x.strip()) - 1 for x in selection.split(",")]
                chosen_roots = [all_roots[i] for i in indices if 0 <= i < len(all_roots)]
            except ValueError:
                print("Invalid selection, defaulting to all locations.")
                chosen_roots = all_roots
        else:
            chosen_roots = all_roots

        if not chosen_roots:
            print("No valid locations selected.")
            continue

        results = run_search(query, chosen_roots, case_sensitive)
        total_found = sum(len(v) for v in results.values())

        if total_found == 0:
            searched_labels = ", ".join(label for _, label in chosen_roots)
            print(f"\n'{query}' was not found in: {searched_labels}")
            remaining = [r for r in all_roots if r not in chosen_roots]
            if remaining:
                retry = input("Would you like to search the remaining locations too? (y/N): ").strip().lower()
                if retry == "y":
                    run_search(query, remaining, case_sensitive)

        again = input("\nSearch again? (y/N): ").strip().lower()
        if again != "y":
            print("Done.")
            break


def find_file(query, case_sensitive=False, roots=None, max_entries_per_dir=None, parallel=True):
    """
    Programmatic entry point (no input() prompts) -- best for notebooks:
        find_file("dataset.csv")
        find_file("PolypGen2021", max_entries_per_dir=15)
    """
    env = detect_environment()
    search_roots = roots if roots else get_search_roots(env)
    if not search_roots:
        print("No accessible search roots were found in this environment.")
        return {}
    return run_search(query, search_roots, case_sensitive, max_entries_per_dir, parallel)


if __name__ == "__main__":
    interactive_search()

Detected environment: KAGGLE

Available search locations:
  [1] Kaggle attached datasets  ->  /kaggle/input
  [2] Kaggle working directory  ->  /kaggle/working



Enter filename or partial filename to search for (or 'quit'):  input
Case-sensitive search? (y/N):  N



Search scope:
  [a] All available locations
  [n] Pick specific location(s) by number


Choose scope (a/n):  a



SEARCHING FOR: 'input'  (case-insensitive)
Mode: parallel (multi-threaded)

--- Kaggle attached datasets (/kaggle/input) ---
  No matches found under /kaggle/input.

--- Kaggle working directory (/kaggle/working) ---
  No matches found under /kaggle/working.
SEARCH COMPLETE: 0 total match(es) across 2 location(s).

'input' was not found in: Kaggle attached datasets, Kaggle working directory


KeyboardInterrupt: Interrupted by user

In [2]:
_print_subtree("/kaggle/input", "")

└── datasets
    └── gokulrocky
        ├── chakramodel-evaluation-datasets
        │   ├── cvc-clinicdb
        │   │   ├── images
        │   │   │   ├── 0000.png
        │   │   │   ├── 0001.png
        │   │   │   ├── 0002.png
        │   │   │   ├── 0003.png
        │   │   │   ├── 0004.png
        │   │   │   ├── 0005.png
        │   │   │   ├── 0006.png
        │   │   │   ├── 0007.png
        │   │   │   ├── 0008.png
        │   │   │   ├── 0009.png
        │   │   │   ├── 0010.png
        │   │   │   ├── 0011.png
        │   │   │   ├── 0012.png
        │   │   │   ├── 0013.png
        │   │   │   ├── 0014.png
        │   │   │   ├── 0015.png
        │   │   │   ├── 0016.png
        │   │   │   ├── 0017.png
        │   │   │   ├── 0018.png
        │   │   │   ├── 0019.png
        │   │   │   ├── 0020.png
        │   │   │   ├── 0021.png
        │   │   │   ├── 0022.png
        │   │   │   ├── 0023.png
        │   │   │   ├── 0024.png
        │   │   │   ├── 0025.png
        │ 